In [ ]:
!pip install -q datasets sentence-transformers faiss-cpu google-generativeai

In [ ]:
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
import pandas as pd
import textwrap
import google.generativeai as genai

In [ ]:
dataset = load_dataset("lavita/MedQuAD")
dataset

In [ ]:
print(dataset)

In [ ]:
df = dataset["train"].to_pandas()

In [ ]:
df.columns

In [ ]:
df[["question", "answer", "document_source"]].isna().sum()

In [ ]:
df = df[["question", "answer", "document_source"]].dropna()
df.head()

In [ ]:
len(df)

In [ ]:
documents = []

for idx, row in df.iterrows():
    text = f"""
Question: {row['question']}

Answer: {row['answer']}
"""

    documents.append({
        "id": idx,
        "text": text,
        "source": row["document_source"],
        "question": row["question"]
    })

len(documents)

In [ ]:
print(documents[0]["text"][:1000])

In [ ]:
embedding_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

In [ ]:
small_docs = documents

texts = [doc["text"] for doc in small_docs]

embeddings = embedding_model.encode(
    texts,
    show_progress_bar=True,
    convert_to_numpy=True
)

embeddings.shape

In [ ]:
dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)
index.add(embeddings)

print("Total vectors stored:", index.ntotal)

In [ ]:
def retrieve_docs(query, top_k=5):
    query_embedding = embedding_model.encode([query], convert_to_numpy=True)

    distances, indices = index.search(query_embedding, top_k)

    results = []
    seen_questions = set()

    for distance, idx in zip(distances[0], indices[0]):
        doc = small_docs[idx]

        question_key = doc["question"].strip().lower()

        if question_key in seen_questions:
            continue

        seen_questions.add(question_key)

        results.append({
            "text": doc["text"],
            "source": doc["source"],
            "question": doc["question"],
            "distance": float(distance)
        })

    return results

In [ ]:
def is_relevant(retrieved_docs, threshold=1.0):
    if len(retrieved_docs) == 0:
        return False

    best_distance = retrieved_docs[0]["distance"]
    return best_distance <= threshold

In [ ]:
query = "What are the symptoms of diabetes?"

results = retrieve_docs(query, top_k=3)

for i, doc in enumerate(results, 1):
    print(f"\n--- Result {i} ---")
    print("Source:", doc["source"])
    print("Matched Question:", doc["question"])
    print("Distance:", doc["distance"])
    print(doc["text"][:700])

In [ ]:
!pip install -q langchain-groq langchain-core

In [ ]:
import os
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage

os.environ["GROQ_API_KEY"] = "Enter your API Key here"


In [ ]:
llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0,
    groq_api_key=os.environ["GROQ_API_KEY"]
)

In [ ]:
def generate_answer(query, retrieved_docs):
    context = "\n\n".join([
        f"[Source {i+1}]\n{doc['text']}"
        for i, doc in enumerate(retrieved_docs)
    ])

    prompt = f"""
You are a medical document assistant.

Answer the user's question using ONLY the provided sources.

Rules:
1. Use only the information from the provided sources.
2. Add citations in the answer using [Source 1], [Source 2], etc.
3. If the sources do not contain enough information, say:
   "The provided medical documents do not contain enough information."
4. Keep the answer simple and clear.
5. Do not diagnose the user.
6. Suggest consulting a healthcare professional when appropriate.

Sources:
{context}

User question:
{query}

Answer with citations:
"""

    response = llm.invoke([HumanMessage(content=prompt)])
    return response.content

In [ ]:
def rag_chatbot(query, top_k=5, threshold=1.0):
    retrieved_docs = retrieve_docs(query, top_k=top_k)

    print("Question:")
    print(query)

    if not is_relevant(retrieved_docs, threshold=threshold):
        print("\nAnswer:")
        print("The provided medical documents do not contain enough information.")

        print("\nReason:")
        if len(retrieved_docs) > 0:
            print(f"Best retrieval distance was {retrieved_docs[0]['distance']}, which is above threshold {threshold}.")
        else:
            print("No documents were retrieved.")
        return

    answer = generate_answer(query, retrieved_docs)

    print("\nAnswer:")
    print(answer)

    print("\nRetrieved Sources:")
    for i, doc in enumerate(retrieved_docs, 1):
        print(f"\n--- Source {i} ---")
        print("Dataset Source:", doc["source"])
        print("Matched Question:", doc["question"])
        print("Distance:", doc["distance"])

In [ ]:
rag_chatbot("How is high blood pressure treated?")

In [ ]:
rag_chatbot("Who won the FIFA World Cup?")

In [ ]:
rag_chatbot("What are the common symptoms of diabetes?")